# Standardization
- clean_sim_data

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, to_date

import sys, os, time, getpass

sys.path.append("/home/tatiane/lib/")
import pessoal
from pessoal import *

spark = SparkSession.builder \
    .appName("SIM Processing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [7]:
BASE_PATH = "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/"

df = spark.read.parquet(f"{BASE_PATH}data/raw/sim/conversion/**/*.parquet")

In [9]:
df[2].count()

TypeError: 'Column' object is not callable

- Analisys

In [ ]:
pessoal.completudeSchema(df)

In [ ]:
pessoal?

- Structured

In [3]:
selected_cols = [
    "IDADE", "SEXO", "RACACOR", "ESTCIV",
    "CODMUNRES", "CODMUNOCOR",
    "DTOBITO", "DTNASC",
    "CAUSABAS", "CAUSABAS_O",
    "ASSISTMED", "NECROPSIA", "LOCOCOR",
    "IDADEMAE", "ESCMAE", "GESTACAO", "PARTO",
    "PESO", "QTDFILVIVO", "QTDFILMORT",
    "OBITOPARTO", "OBITOPUERP"
]

df = df.select(*selected_cols)

Processando: Mortalidade_Geral_2018_00001.xml
Processando: Mortalidade_Geral_2018_00002.xml
Processando: Mortalidade_Geral_2018_00003.xml
Processando: Mortalidade_Geral_2018_00004.xml
Processando: Mortalidade_Geral_2018_00005.xml
Processando: Mortalidade_Geral_2018_00006.xml
Processando: Mortalidade_Geral_2018_00007.xml


In [5]:
df = df.withColumn("DTOBITO", to_date(col("DTOBITO"))) \
       .withColumn("DTNASC", to_date(col("DTNASC"))) \
       .withColumn("ano", year(col("DTOBITO")))

In [6]:
df = df.filter(col("DTOBITO").isNotNull())

In [ ]:
from pyspark.sql.functions import col

anos = [2018, 2019, 2020, 2021, 2022]

for ano in anos:
    df_ano = df.filter(col("ano") == ano)
    
    df_ano.coalesce(1) \
        .write \
        .mode("overwrite") \
        .parquet(f"/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/analytics/sim_{ano}")